In [10]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/media_recommender/data', exist_ok=True)
!cp -r /content/drive/MyDrive/media_recommender_data/* /content/media_recommender/data/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Content Embeddings — Weighted Tag Vectors

Builds a content-based similarity signal from AniList genre/tag data, used for **cold-start handling** — representing anime by what they're about rather than who liked them, so titles with little or no rating history still have usable features.

**Approach note:** an earlier version of this step used sentence-transformer embeddings on plot synopses (and later, synopsis + genre/tag text combined) to measure similarity. That approach under-performed on real test cases — e.g. it rated *Mob Psycho 100* as more similar to *Attack on Titan* than *Fullmetal Alchemist: Brotherhood* was, which doesn't match genre-savvy fan intuition. The issue: raw synopsis text captures plot events, not tone/theme, and even tag-enriched text over-weighted common, low-signal tags ("Male Protagonist", "Shounen") while diluting rare, meaningful ones ("Cannibalism", "Steampunk") on titles with long tag lists. The approach below instead builds explicit **rarity-weighted tag vectors** (an IDF-style scheme), which directly fixes both problems.

In [19]:
import json
import os
import math
from collections import Counter
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
import shutil

## Load anime metadata

Comes from `01_data_collection.ipynb`'s AniList pull.

In [12]:
DATA_DIR = '/content/media_recommender/data'

with open(os.path.join(DATA_DIR, "anime_data.jsonl"), "r") as f:
    anime_content = [json.loads(line) for line in f]

5000 5000


## Build rarity-weighted tag vectors

Each anime is represented as a vector over the tag vocabulary, where each tag's weight is scaled by how rare it is (an IDF-style scheme: `log(total_items / (times_this_tag_appears + 1))`).

**Why rarity weighting, not raw tag presence:** a tag like "Male Protagonist" appears on a huge fraction of all anime and tells you almost nothing when shared between two titles. A tag like "Cannibalism" or "Steampunk" is rare and genuinely indicates deep similarity when shared. Raw (unweighted) tag overlap treats both the same, and also gets diluted on titles with unusually long tag lists (more total tags to add unrelated noise). Rarity weighting fixes both issues at once.

In [ ]:
tag_counts = Counter()
for item in anime_content:
    for t in item.get('tags', []):
        tag_counts[t['name']] += 1

total_items = len(anime_content)

def tag_weight(tag_name):
    # Rare tags get a high weight, common tags get pushed toward ~0
    return math.log(total_items / (tag_counts[tag_name] + 1))

all_tags = sorted(tag_counts.keys())
tag_to_col = {t: i for i, t in enumerate(all_tags)}

def build_tag_vector(item):
    vec = np.zeros(len(all_tags))
    for t in item.get('tags', []):
        vec[tag_to_col[t['name']]] = tag_weight(t['name'])
    return vec

anime_tag_vectors = np.array([build_tag_vector(a) for a in anime_content])

print(anime_tag_vectors.shape)

np.save(os.path.join(DATA_DIR, "anime_tag_vectors.npy"), anime_tag_vectors)

In [ ]:
shutil.copytree('/content/media_recommender/data', '/content/drive/MyDrive/media_recommender_data', dirs_exist_ok=True)
print(os.listdir('/content/drive/MyDrive/media_recommender_data'))